<a href="https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## **1. Two paper findings + my methodology questions**

**Finding #4 — "The Freshness Multiplier"**

**Claim**: Content aged 365+ days that was refreshed within the last 30 days shows a 3.2x health score boost and a 57x impression increase, compared to similarly old, unrefreshed content. The paper calls refresh timing "one of the strongest measured levers available."

**Where does the label come from?** The "refreshed" label appears to come from a simple date comparison: content whose last-update timestamp falls within the past 30 days. This is a clean, observable label, not a derived or composite score, which is a strength. However, the paper doesn't say why a given page was refreshed, whether refresh candidates were selected randomly, or hand-picked by an editor who may have already judged the page worth saving.

**Does the validation design carry the claim?** This is a before/after
group comparison, not a held-out validation split there's no train/test structure here at all, which is appropriate for a descriptive comparison but not sufficient to support the causal-sounding phrase "one of the strongest measured levers." Without knowing how refresh candidates were selected, the comparison can't rule out selection bias: pages chosen for refresh may have already had recovery potential before the refresh happened, meaning the validation design as described doesn't fully carry the strength of the claim's phrasing, even though the underlying pattern is likely real and useful directionally.


**Finding #10 — "AI Model Performance" (OpenAI vs. Gemini)**

**Claim**: An age-controlled comparison shows performance differences between OpenAI-authored and Gemini-authored content, with each model family leading in different age cohorts.

**Where does the label come from?** The "OpenAI" vs. "Gemini" label presumably comes from metadata tracking which AI tool produced each piece of content; a reasonably direct, observable label rather than something derived or inferred. But the paper doesn't explain whether every brand used both tools, or whether some brands used exclusively one, meaning the model label may be entangled with a brand label without a way to separate the two.

**Does the validation design carry the claim?** The comparison controls for content age (comparing within the same publication window), which is a meaningful and appropriate control. However, it doesn't control for brand, and since brands likely self-selected which AI tool to use, any performance difference between the two groups could reflect brand-level factors (audience, strategy, topic focus) rather than a genuine model effect. The paper's own cautious framing ("read as a process comparison, not a victory lap for one provider family") suggests the authors were aware of this limitation, but the underlying design comparing two groups without controlling for the brand doing the choosing doesn't fully carry a claim about the AI models themselves.

In [1]:
# Section 1 — research paper audit: label source + validation design

paper_audit = {
    "Finding #4 (Freshness Multiplier)": {
        "claim": "365+ day content refreshed within 30 days shows 3.2x health boost, 57x impressions",
        "label_source": "Simple date-based flag (updated within last 30 days) — clean, but selection process for WHO gets refreshed is undisclosed",
        "validation_design": "Before/after group comparison, no train/test split — cannot rule out selection bias in which pages were chosen for refresh"
    },
    "Finding #10 (OpenAI vs Gemini)": {
        "claim": "Age-controlled comparison shows performance differences between OpenAI and Gemini authored content",
        "label_source": "Content-level metadata tag for which AI tool authored the piece — observable, but entangled with brand identity",
        "validation_design": "Controls for content age, but not for brand — brand self-selection of AI tool is a plausible confound not addressed by the design"
    }
}

for finding, details in paper_audit.items():
    print(f"\n{finding}")
    for key, value in details.items():
        print(f"  {key}: {value}")


Finding #4 (Freshness Multiplier)
  claim: 365+ day content refreshed within 30 days shows 3.2x health boost, 57x impressions
  label_source: Simple date-based flag (updated within last 30 days) — clean, but selection process for WHO gets refreshed is undisclosed
  validation_design: Before/after group comparison, no train/test split — cannot rule out selection bias in which pages were chosen for refresh

Finding #10 (OpenAI vs Gemini)
  claim: Age-controlled comparison shows performance differences between OpenAI and Gemini authored content
  label_source: Content-level metadata tag for which AI tool authored the piece — observable, but entangled with brand identity
  validation_design: Controls for content age, but not for brand — brand self-selection of AI tool is a plausible confound not addressed by the design


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

To test whether my Week 5 model's use of a client-grouped split genuinely mattered, I re-trained the same Logistic Regression model with the same features, same target and under two different split strategies and compared the resulting AUC scores directly.

Before (naive random split): AUC = 0.595. This split ignored client boundaries entirely, allowing pages from the same client to land in both the training and test sets.

After (grouped by client split): AUC = 0.557. This is the same approach used in my Week 5 notebook, where GroupShuffleSplit ensured every client's pages stayed entirely within either the training set or the test set, never both.

The naive split produced a higher score than the grouped split with a difference of 0.038 AUC. This confirms the concern raised back in Week 5: when pages from the same client appear in both training and testing, the model can partially learn client-specific patterns during training and then benefit from recognizing those same patterns at test time, inflating the evaluation score without reflecting genuine predictive skill on unseen clients.

The grouped split's score of 0.557 should be treated as the more honest estimate of real-world performance, since it evaluates the model exclusively on clients it has never encountered during training — the scenario that actually matters when deciding whether this model would generalize to a new client's content going forward. The 0.038-point gap, while modest in absolute terms, is a concrete, measured demonstration of exactly the risk a naive split introduces.

In [ ]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git
%cd flyrank-ml-internship

In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review"] = (df["trend_direction"] == "down").astype(int)

model_features = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                   "engagement_rate", "content_age_days"]

In [4]:
X = df[model_features].fillna(0)
y = df["needs_review"]

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_naive = LogisticRegression(max_iter=1000)
model_naive.fit(X_train_naive, y_train_naive)

naive_score = roc_auc_score(y_test_naive, model_naive.predict_proba(X_test_naive)[:, 1])
print(f"BEFORE (naive random split) AUC: {naive_score:.3f}")

BEFORE (naive random split) AUC: 0.595


In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train_grouped = train_df[model_features].fillna(0)
y_train_grouped = train_df["needs_review"]
X_test_grouped = test_df[model_features].fillna(0)
y_test_grouped = test_df["needs_review"]

model_grouped = LogisticRegression(max_iter=1000)
model_grouped.fit(X_train_grouped, y_train_grouped)

grouped_score = roc_auc_score(y_test_grouped, model_grouped.predict_proba(X_test_grouped)[:, 1])
print(f"AFTER (grouped by client) AUC: {grouped_score:.3f}")

print(f"\nDifference: {naive_score:.3f} -> {grouped_score:.3f}")

AFTER (grouped by client) AUC: 0.557

Difference: 0.595 -> 0.557


In [6]:
# Section 2 — before/after comparison summary

comparison = pd.DataFrame({
    "Split Method": ["Naive random split (BEFORE)", "Grouped by client (AFTER)"],
    "AUC": [naive_score, grouped_score]
})

print(comparison.to_string(index=False))
print(f"\nDifference: {naive_score - grouped_score:.3f} AUC points")
print("The naive split overestimates performance by allowing client-specific")
print("patterns to leak between train and test sets.")

               Split Method      AUC
Naive random split (BEFORE) 0.594661
  Grouped by client (AFTER) 0.557426

Difference: 0.037 AUC points
The naive split overestimates performance by allowing client-specific
patterns to leak between train and test sets.


## **3. Leakage audit**

*The same hunt from Week 3, on your final feature set.*

I repeated the leakage audit from Week 3, this time applying it directly to my final Week 5 feature set and grouped validation. My six model features were impressions_90d, clicks_90d, ctr, avg_position, engagement_rate, and content_age_days. I checked these against the known forbidden signals trend_direction, trend_pct, health_score, priority_score, action_type, and is_declining_label and confirmed that none of them were used in the final model.

To test the audit under real conditions rather than just naming what I avoided, I deliberately added trend_pct; a signal derived from the same underlying trend information used to construct my target as a seventh feature. The honest six-feature model produced AUC = 0.557 under the grouped client split, while the leaky seven-feature model produced AUC = 1.000. The jump from 0.557 to a perfect 1.000 is a clear leakage signature rather than evidence of genuine model improvement — no real prediction task should ever score perfectly.

This confirms that trend_pct is a genuine leakage risk, consistent with why it was excluded from my model from the start. It also demonstrates something more useful than the exclusion itself: if trend_pct were ever accidentally included in a future version of this model, the resulting near-perfect score would serve as an immediate, visible warning sign not something that could quietly slip through unnoticed.

In [7]:
# Leakage audit — forbidden signals check

model_features = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                   "engagement_rate", "content_age_days"]

forbidden_signals = ["trend_direction", "trend_pct", "health_score",
                      "priority_score", "action_type", "is_declining_label"]

print("Final model features:")
print(model_features)
print()
print("Checking against forbidden signals:")
for signal in forbidden_signals:
    status = "LEAKED — PROBLEM" if signal in model_features else "not used — clean"
    print(f"  {signal}: {status}")

Final model features:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days']

Checking against forbidden signals:
  trend_direction: not used — clean
  trend_pct: not used — clean
  health_score: not used — clean
  priority_score: not used — clean
  action_type: not used — clean
  is_declining_label: not used — clean


# **4. Claim rewrite**

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original**: “This means that of the top 50 pages the model ranked highest, 37 genuinely needed review, compared to 23 for the baseline rule.”

**Rewritten**: “Among the top 50 pages ranked by the model, 37 were observed to match the needs_review outcome, compared with 23 for the baseline rule. This suggests that the model provided stronger directional prioritization of review candidates in this evaluation, but it should be treated as decision-support rather than proof that those pages genuinely required review.”

In [9]:
# Section 4 — claim rewrite

original_claim = (
    "This means that of the top 50 pages the model ranked highest, "
    "37 genuinely needed review, compared to 23 for the baseline rule."
)

rewritten_claim = (
    "Among the top 50 pages ranked by the model, 37 were observed to match "
    "the needs_review outcome, compared with 23 for the baseline rule. This "
    "suggests that the model provided stronger directional prioritization of "
    "review candidates in this evaluation, but it should be treated as "
    "decision-support rather than proof that those pages genuinely required review."
)

print("ORIGINAL (Week 5):")
print(original_claim)
print("\nREWRITTEN (safe language):")
print(rewritten_claim)
print("\nKey change: 'genuinely needed review' (asserted as fact) -> 'observed to")
print("match the needs_review outcome' (acknowledges this is a proxy label, not")
print("verified ground truth)")

ORIGINAL (Week 5):
This means that of the top 50 pages the model ranked highest, 37 genuinely needed review, compared to 23 for the baseline rule.

REWRITTEN (safe language):
Among the top 50 pages ranked by the model, 37 were observed to match the needs_review outcome, compared with 23 for the baseline rule. This suggests that the model provided stronger directional prioritization of review candidates in this evaluation, but it should be treated as decision-support rather than proof that those pages genuinely required review.

Key change: 'genuinely needed review' (asserted as fact) -> 'observed to
match the needs_review outcome' (acknowledges this is a proxy label, not
verified ground truth)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.